In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import cv2
from astropy.io import fits
import os

%matplotlib widget

In [ ]:
def calculate_laplacian_focus(image, type=cv2.CV_32F):
    laplacian = cv2.Laplacian(image, type)
    variance = laplacian.var()
    return variance, laplacian

def calculate_brenner_measure(image):
    """
    Brenner's focus measure.

    Parameters
    ----------
    image : np.ndarray
        The input image in grayscale.
    Returns
    -------
    float
        The Brenner value.
    """

    # Get the dimensions of the image
    height, width = image.shape

    # Initialize two matrices for horizontal and vertical focus measures
    horizontal_diff = np.zeros((height, width))
    vertical_diff = np.zeros((height, width))

    # Calculate horizontal and vertical focus measures
    horizontal_diff[:, : width - 2] = np.clip(
        image[:, 2:] - image[:, :-2], 0, None
    )
    vertical_diff[: height - 2, :] = np.clip(
        image[2:, :] - image[:-2, :], 0, None
    )

    # Calculate final focus measure
    Brenner_measure = np.max((horizontal_diff, vertical_diff), axis=0) ** 2

    # Convert focus measure matrix to 8-bit for visualization
    #Brenner_image = ((Brenner_measure / Brenner_measure.max()) * 255).astype(np.uint8)

    return Brenner_measure.mean()

# Play with a synthetic image

In [ ]:
# Create a synthetic image with a bright spot
image_synthetic = np.random.rand(1000, 1000)
image_synthetic[200:300, 500:600] = 3
image_synthetic = signal.convolve2d(image_synthetic, np.ones((10, 10)), mode='same', boundary='symm')

plt.figure(figsize=(4, 4))
plt.imshow(image_synthetic, origin='lower', cmap='gray')
print(image_synthetic.shape)

In [ ]:
variance, laplacian = calculate_laplacian_focus(image_synthetic, type=cv2.CV_64F)
print(variance)
Brenner_measure = calculate_brenner_measure(image_synthetic)
print(Brenner_measure)

In [ ]:
all_brenner_measures = []
for i in range(10):
    image_synthetic_blurred = signal.convolve2d(image_synthetic, np.ones((i+1, i+1))*10, mode='same', boundary='symm')
    plt.figure(figsize=(4, 4))
    plt.imshow(image_synthetic_blurred, origin='lower', cmap='gray')
    variance_blurred, _ = calculate_laplacian_focus(image_synthetic_blurred, type=cv2.CV_64F)
    Brenner_measure_blurred = calculate_brenner_measure(image_synthetic_blurred)
    all_brenner_measures.append(Brenner_measure_blurred)
    print(f"Blur level {i+1}: Variance = {variance_blurred}, Brenner Measure = {Brenner_measure_blurred}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(all_brenner_measures, marker='o')
plt.title("Brenner Measure vs Blur Level")
plt.ylabel("Brenner Measure")
plt.grid()
plt.show()

# Get real data

In [ ]:
data_dir = '/home/lmousset/Projets_Recherche/LSST/20260423_focus_empty_empty/'

img_numbers = np.arange(45, 59)
focuser = np.concatenate([np.arange(72, 83)*1000, np.array([85000, 76500, 77500])])
print(img_numbers)
print(focuser)

dict_images = {}
for i, img_num in enumerate(img_numbers):
    dict_images[img_num] = focuser[i]
    print(img_num, dict_images[img_num])


In [ ]:
fits_files = [f for f in os.listdir(data_dir) if f.endswith('.fits')]

print(fits_files)
nimages = len(fits_files)
print(f"Number of FITS files: {nimages}")

# Get the index of each image from the filename and sort the files based on that index
for f in fits_files:
    index_img = int(f[-7:-5])
    print(f, index_img)

fits_files_sorted = sorted(fits_files, key=lambda x: int(x[-7:-5]))
print(fits_files_sorted)

In [ ]:
# Open a FITS file and display the image
i = 5 # Index of the file
hdu = fits.open(os.path.join(data_dir, fits_files_sorted[i]))
hdu.info()
image = hdu[1].data
print(image.shape)

plt.figure(figsize=(4, 4))
plt.title(f"Image {img_numbers[i]} with focuser {dict_images[img_numbers[i]]}")
plt.imshow(image, origin='lower', cmap='viridis', vmin=0, vmax=600)
plt.colorbar()
plt.tight_layout()

In [ ]:
xspot, yspot = (1472, 1767)
print(xspot, yspot)
image_cut = image[yspot-500:yspot+500, xspot-500:xspot+500]
plt.figure(figsize=(4, 4))
plt.imshow(image_cut, origin='lower', cmap='viridis', vmin=0, vmax=600)
plt.colorbar()
plt.tight_layout()

In [ ]:
# Plot all images with the center of the spot marked
all_center_spots = []
all_images_cut = []
fig, axs = plt.subplots(nrows=4, ncols=4, figsize=(12, 12))
axs = axs.flatten()
for i in range(nimages):
    print(f"Image {img_numbers[i]}: Focuser position = {focuser[i]}")
    hdu = fits.open(os.path.join(data_dir, fits_files_sorted[i]))
    image = hdu[1].data

    dpix = 300
    image_cut = image[yspot-dpix:yspot+dpix, xspot-dpix:xspot+dpix]
    all_images_cut.append(image_cut)

    # Center of the spot on the cut image
    center_spot = np.argmax(signal.convolve2d(image_cut, np.ones((20, 20)), mode='same', boundary='symm'))
    xcenter = center_spot % (2*dpix)
    ycenter = center_spot // (2*dpix)
    all_center_spots.append((xcenter, ycenter))
    #image_cut_norm = image_cut - np.median(image_cut)
    
    axs[i].set_title(f"Image {img_numbers[i]}: Focuser = {dict_images[img_numbers[i]]}")
    axs[i].imshow(image_cut, origin='lower', cmap='viridis', vmin=0, vmax=600)
    axs[i].plot(xcenter, ycenter, 'ro')  # Mark the center of the spot
    
fig.tight_layout()

In [ ]:
# Position of the center of the spot vs image index
all_center_spots = np.array(all_center_spots)
print(all_center_spots)

plt.figure(figsize=(10, 5))
plt.plot(all_center_spots[:, 0], label='X center of spot')
plt.plot(all_center_spots[:, 1], label='Y center of spot')
plt.xlabel('Image Index')
plt.ylabel('Pixel Position')
plt.title('Center of Spot Position vs Image Index')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Image cut at the center of the spot for all images
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs = axs.flatten()
for i in range(nimages):
    xcenter = all_center_spots[i][0]
    ycenter = all_center_spots[i][1]
    axs[0].plot(all_images_cut[i][xcenter, :], label=f"Focuser = {dict_images[img_numbers[i]]}")
    axs[1].plot(all_images_cut[i][:, ycenter], label=f"Focuser = {dict_images[img_numbers[i]]}")

#axs[0].legend()
#axs[1].legend()
plt.grid()
plt.show()

In [ ]:
allBrenner_measures = []
allVar_Laplacian = []
for i in range(nimages):
    hdu = fits.open(os.path.join(data_dir, fits_files[i]))
    image = hdu[1].data

    dpix = 300
    image_cut = image[yspot-dpix:yspot+dpix, xspot-dpix:xspot+dpix]

    Brenner_measure = calculate_brenner_measure(image_cut)
    allBrenner_measures.append(Brenner_measure)

    variance, _ = calculate_laplacian_focus(image_cut)
    allVar_Laplacian.append(variance)

    

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(focuser, allBrenner_measures, 'o')
plt.title("Brenner Measure vs Focuser Position")
plt.xlabel("Focuser Position")
plt.ylabel("Brenner Measure")
plt.grid()
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(focuser, allVar_Laplacian, 'o')
plt.title("Laplacian Variance vs Focuser Position")
plt.xlabel("Focuser Position")
plt.ylabel("Laplacian Variance")
plt.grid()
plt.tight_layout()

In [ ]:
focuser

In [ ]:
# Ensure arrays
x = np.asarray(focuser, dtype=float)
y = np.asarray(allVar_Laplacian, dtype=float)

# Indices to exclude
exclude_idx = [0, 1, 2, 3, -1, -3]

mask = np.ones(len(x), dtype=bool)
mask[exclude_idx] = False

x_fit_data = x[mask]
y_fit_data = y[mask]

# Quadratic fit (common for focus curves near optimum)
coef = np.polyfit(x_fit_data, y_fit_data, 2)   # y = a x^2 + b x + c
a, b, c = coef

# Best focus from parabola minimum: x = -b/(2a)
x_best = -b / (2 * a)
y_best = np.polyval(coef, x_best)

# Smooth curve for plotting
x_line = np.linspace(x.min(), x.max(), 400)
y_line = np.polyval(coef, x_line)

plt.figure(figsize=(8, 5))
plt.plot(x[mask], y[mask], "o", label="Used points")
plt.plot(x[~mask], y[~mask], "rx", ms=10, label="Excluded points")
plt.plot(x_line, y_line, "-", label="Quadratic fit")
plt.axvline(x_best, ls="--", label=f"Best focus ~ {x_best:.1f}")
plt.xlabel("Focuser Position")
plt.ylabel("Laplacian Variance")
plt.grid(True)
plt.legend()
plt.tight_layout()
print(f"Fit coefficients: a={a:.3e}, b={b:.3e}, c={c:.3e}")
print(f"Estimated best focus position: {x_best:.2f}")